In [ ]:
using LinearAlgebra
using PythonPlot

In [ ]:
# Dynamics
function pendulum_dynamics(x, u)
    gravity = 9.81
    length = 1.0
    mass = 1.0

    θ = x[1]
    θ̇ = x[2]

    ẋ = [θ̇;
        -(gravity/length)*sin(θ) + u/(mass*length*length)]
    
    return ẋ
end

function pendulum_step(x, u)
    Δt = 0.1
    xm = x + 0.5*Δt*pendulum_dynamics(x,u)
    return x + Δt*pendulum_dynamics(xm,u)
end

In [ ]:
# Dynamics Jacobians
function pendulum_deriv(x, u)
    gravity = 9.81
    length = 1.0
    mass = 1.0

    θ = x[1]
    θ̇ = x[2]

    Ac = [0                       1.0;
         -(gravity/length)*cos(θ)  0]
    
    return Ac
end

function pendulum_step_deriv(x, u)
    Δt = 0.1
    xm = x + 0.5*Δt*pendulum_dynamics(x,u)

    A = I(2) + Δt*pendulum_deriv(xm,u)*(I(2) + 0.5*Δt*pendulum_deriv(x,u))
    
    return A
end

In [ ]:
# Measurements: we can see the tip of the pendulum
function tip(x)
    length = 1.0
    θ = x[1]
    return [length*sin(π-θ); length*cos(π-θ)]
end

function tip_deriv(x)
    length = 1.0
    θ = x[1]
    C = [-length*cos(π-θ) 0;
         length*sin(π-θ)  0]
    return C
end

In [ ]:
# Noise
V = 0.0001*I(2) #process noise
W = 0.1*I(2) #measurement noise

In [ ]:
# Simulation
Tfinal = 10
Nt = 101
xtraj = zeros(2, Nt)
xtraj[:,1] = [2.5; 0]

ytraj = zeros(2, Nt)
ytraj[:,1] = tip(xtraj[:,1])

for k = 1:(Nt-1)
    xtraj[:,k+1] = pendulum_step(xtraj[:,k], 0) + sqrt(V)*randn(2)
    ytraj[:,k+1] = tip(xtraj[:,k+1]) + sqrt(W)*randn(2)
end

In [ ]:
# Let's look at the state trajectory
times = LinRange(0,Tfinal,Nt)
plot(times, xtraj[1,:], label='q')
plot(times, xtraj[2,:], label='v')
legend()

In [ ]:
# Let's look at the measurements
plot(times, ytraj[1,:])
plot(times, ytraj[2,:])

In [ ]:
# Filter Initialization
xfilt = zeros(2, Nt)
xfilt[:,1] = xtraj[:,1] + sqrt(W)*randn(2)

P = zeros(2,2,Nt)
P[:,:,1] .= W

In [ ]:
for k = 1:(Nt-1)
    #Prediction
    xpred = pendulum_step(xfilt[:,k], 0)
    A = pendulum_step_deriv(xfilt[:,k], 0)
    Ppred = A*P[:,:,k]*A' + V

    #Innovation
    z = ytraj[:,k+1] - tip(xpred)
    C = tip_deriv(xpred)
    S = C*Ppred*C' + W

    #Kalman Gain
    L = Ppred*C'/S

    #Update
    xfilt[:,k+1] = xpred + L*z
    
    P[:,:,k+1] = (I(2) - L*C)*Ppred*(I(2) - L*C)' + L*V*L'
end

In [ ]:
# Plot the estimate vs. the ground truth
plot(times, xtraj[1,:], label="qtrue")
plot(times, xfilt[1,:], label="qfilt")
legend()

In [ ]:
plot(times, xtraj[2,:], label="vtrue")
plot(times, xfilt[2,:], label="vfilt")
legend()